<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC Spot Grid Trading Backtest

This notebook is organized in stages so the grid logic can be validated against the **KZM Grid Template** before running a historical backtest.

**Workflow:** Setup → Historical Data → Excel Grid Model → Backtest Engine → Results

## 1. Setup

In [ ]:
import os
import io
import glob
import datetime

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Single source of truth for project data
DATA_DIR = "/content/drive/MyDrive/03.Trading/00.Live Trading"

print(f"Data directory: {DATA_DIR}")

## 2. Configuration

In [ ]:
# Market data configuration
SYMBOL = "BTCUSDT"
START_DATE = "2024-01-01"
END_DATE = "2026-01-01"
TIMEFRAMES = ["1m", "1h", "1d"]

# KZM Excel-template parameters
GRID_CAPITAL = 3000.0
GRID_CEILING = 8987.0
GRID_FLOOR = 1987.0
GRID_GAP = 70.0

BUY_FEE = 0.001   # 0.1%
SELL_FEE = 0.001  # 0.1%

## 3. Historical Data

The download/combine functions are kept as utilities. They do **not** run automatically, so restarting Colab will not re-download all Binance data.

In [ ]:
def download_binance_history(
    symbol,
    start_date_str,
    end_date_str,
    timeframes,
    data_dir,
):
    """Download Binance Spot monthly kline archives and save each month as CSV."""
    start_date = datetime.datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.datetime.strptime(end_date_str, "%Y-%m-%d")

    os.makedirs(data_dir, exist_ok=True)

    current_date = start_date

    while current_date < end_date:
        year = current_date.year
        month = current_date.month

        for timeframe in timeframes:
            url = (
                "https://data.binance.vision/data/spot/monthly/klines/"
                f"{symbol}/{timeframe}/"
                f"{symbol}-{timeframe}-{year}-{month:02d}.zip"
            )

            try:
                print(f"Downloading {symbol} {timeframe} for {year}-{month:02d}...")

                response = requests.get(url, timeout=60)
                response.raise_for_status()

                columns = [
                    "open_time",
                    "open",
                    "high",
                    "low",
                    "close",
                    "volume",
                    "close_time",
                    "quote_volume",
                    "number_of_trades",
                    "taker_buy_base",
                    "taker_buy_quote",
                    "ignore",
                ]

                df_month = pd.read_csv(
                    io.BytesIO(response.content),
                    compression="zip",
                    header=None,
                    names=columns,
                )

                # Binance archive timestamp format changed for newer archives.
                time_unit = "us" if year >= 2025 else "ms"

                df_month["open_time"] = pd.to_datetime(
                    df_month["open_time"], unit=time_unit, utc=True
                )
                df_month["close_time"] = pd.to_datetime(
                    df_month["close_time"], unit=time_unit, utc=True
                )

                numeric_cols = ["open", "high", "low", "close", "volume"]
                df_month[numeric_cols] = df_month[numeric_cols].astype(float)

                save_path = os.path.join(
                    data_dir,
                    f"{symbol}-{timeframe}-{year}-{month:02d}.csv",
                )
                df_month.to_csv(save_path, index=False)
                print(f"Saved: {save_path}")

            except Exception as exc:
                print(
                    f"Could not download {symbol} {timeframe} "
                    f"{year}-{month:02d}: {exc}"
                )

        if current_date.month == 12:
            current_date = current_date.replace(
                year=current_date.year + 1,
                month=1,
            )
        else:
            current_date = current_date.replace(
                month=current_date.month + 1,
            )

In [ ]:
def combine_monthly_csv(symbol, timeframes, data_dir):
    """Combine monthly CSV files into one chronologically sorted CSV per timeframe."""
    output_paths = {}

    for timeframe in timeframes:
        search_pattern = os.path.join(
            data_dir,
            f"{symbol}-{timeframe}-????-??.csv",
        )
        file_list = sorted(glob.glob(search_pattern))

        if not file_list:
            print(f"No monthly files found for timeframe: {timeframe}")
            continue

        print(f"Combining {len(file_list)} files for {timeframe}...")

        combined_df = pd.concat(
            [pd.read_csv(path) for path in file_list],
            ignore_index=True,
        )

        combined_df["open_time"] = pd.to_datetime(
            combined_df["open_time"],
            utc=True,
        )
        combined_df = (
            combined_df
            .drop_duplicates(subset="open_time")
            .sort_values("open_time")
            .reset_index(drop=True)
        )

        output_path = os.path.join(
            data_dir,
            f"{symbol}-{timeframe}-combined.csv",
        )
        combined_df.to_csv(output_path, index=False)

        output_paths[timeframe] = output_path
        print(f"Created: {output_path} ({len(combined_df):,} rows)")

    return output_paths

In [ ]:
def load_market_data(symbol, timeframe, data_dir):
    """Load one combined market-data CSV and perform basic type cleaning."""
    file_path = os.path.join(
        data_dir,
        f"{symbol}-{timeframe}-combined.csv",
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Combined data file not found: {file_path}\n"
            "Run download_binance_history() and combine_monthly_csv() first."
        )

    df = pd.read_csv(file_path)

    df["open_time"] = pd.to_datetime(df["open_time"], utc=True)

    numeric_cols = ["open", "high", "low", "close", "volume"]
    df[numeric_cols] = df[numeric_cols].astype(float)

    df = (
        df
        .drop_duplicates(subset="open_time")
        .sort_values("open_time")
        .reset_index(drop=True)
    )

    return df


# Primary dataset for the first backtest version
df_1m = load_market_data(SYMBOL, "1m", DATA_DIR)
df_1m.head()

In [ ]:
def validate_market_data(df):
    """Quick integrity check before using the data in a backtest."""
    ohlc_cols = ["open", "high", "low", "close"]

    print(f"Rows              : {len(df):,}")
    print(f"Start             : {df['open_time'].min()}")
    print(f"End               : {df['open_time'].max()}")
    print(f"Duplicate times   : {df['open_time'].duplicated().sum():,}")
    print(
        "Rows missing OHLC : "
        f"{df[ohlc_cols].isna().any(axis=1).sum():,}"
    )


validate_market_data(df_1m)

## 4. Excel Grid Model

Before using historical candles, this section reproduces the arithmetic-grid calculation in the KZM Excel template.

For each grid pair:

- buy at the lower grid price,
- allocate the same quote capital to each level,
- deduct the buy fee from the base asset received,
- sell one grid higher,
- deduct the sell fee from the quote proceeds.

In [ ]:
def build_excel_grid_table(
    capital,
    ceiling,
    floor,
    gap,
    buy_fee=0.001,
    sell_fee=0.001,
):
    """Reproduce the fixed-size arithmetic-grid calculation used in the Excel template."""
    if capital <= 0:
        raise ValueError("capital must be greater than 0.")
    if ceiling <= floor:
        raise ValueError("ceiling must be greater than floor.")
    if gap <= 0:
        raise ValueError("gap must be greater than 0.")

    raw_levels = (ceiling - floor) / gap

    if not np.isclose(raw_levels, round(raw_levels)):
        raise ValueError(
            "(ceiling - floor) must be exactly divisible by gap "
            "for the Excel-template version."
        )

    n_levels = int(round(raw_levels))
    capital_per_level = capital / n_levels

    buy_prices = ceiling - gap * np.arange(1, n_levels + 1)
    sell_prices = buy_prices + gap

    gross_base_amount = capital_per_level / buy_prices
    buy_fee_base = gross_base_amount * buy_fee
    base_amount = gross_base_amount - buy_fee_base

    gross_sell = base_amount * sell_prices
    sell_fee_quote = gross_sell * sell_fee
    net_sell = gross_sell - sell_fee_quote
    profit = net_sell - capital_per_level

    grid = pd.DataFrame(
        {
            "level": np.arange(1, n_levels + 1),
            "buy_price": buy_prices,
            "sell_price": sell_prices,
            "capital_per_level": capital_per_level,
            "gross_base_amount": gross_base_amount,
            "buy_fee_base": buy_fee_base,
            "base_amount": base_amount,
            "gross_sell": gross_sell,
            "sell_fee_quote": sell_fee_quote,
            "net_sell": net_sell,
            "profit": profit,
        }
    )

    return grid


df_grid_excel = build_excel_grid_table(
    capital=GRID_CAPITAL,
    ceiling=GRID_CEILING,
    floor=GRID_FLOOR,
    gap=GRID_GAP,
    buy_fee=BUY_FEE,
    sell_fee=SELL_FEE,
)

print(f"Number of grid levels : {len(df_grid_excel)}")
print(f"Capital per level     : {df_grid_excel['capital_per_level'].iloc[0]:.2f}")
df_grid_excel.head()

In [ ]:
# Checkpoint: first Excel grid pair should be 8,917 -> 8,987
excel_check = df_grid_excel.iloc[0]

print(f"Buy price  : {excel_check['buy_price']:.2f}")
print(f"Sell price : {excel_check['sell_price']:.2f}")
print(f"Base amount: {excel_check['base_amount']:.9f}")
print(f"Profit     : {excel_check['profit']:.6f} USDT")

EXPECTED_FIRST_PROFIT = 0.175064

assert np.isclose(
    excel_check["profit"],
    EXPECTED_FIRST_PROFIT,
    atol=1e-6,
), "Excel replication checkpoint failed."

print("Excel replication checkpoint: PASSED")

## 5. Backtest Engine

Next step: define the candle execution rules (including what happens when one 1-minute candle crosses multiple grid levels) before implementing the historical backtest.

Keeping this section separate prevents us from mixing **grid-calculation logic** with **execution assumptions**.

## 6. Trade Log & Performance

Planned outputs:

- completed grid cycles,
- buy/sell fees,
- realized grid profit,
- open inventory,
- cash balance,
- mark-to-market portfolio value,
- net return,
- maximum drawdown,
- Calmar ratio.